In [ ]:
import numpy as np
import pandas as pd
import tensorflow_probability as tfp
import tensorflow as tf
from meridian import constants
from meridian.data import load
from meridian.model import model, spec, prior_distribution
from meridian.analysis import analyzer as mmm_analyzer

import os, warnings, logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)


from mmm_lab.data_generation.baseline import generate_baseline_geo_data
from mmm_lab.data_generation.marketing import add_marketing_effects

np.random.seed(42)
baseline_df = generate_baseline_geo_data(n_geos=40, n_weeks=104, start_date='2023-01-01')
df = add_marketing_effects(baseline_df, channels=['tv', 'paid_search'])

df_sorted = df.sort_values(['geo', 'date'])
n_geos = df['geo'].nunique()
n_time  = df['date'].nunique()
y_matrix = df_sorted['total_bookings'].values.reshape(n_geos, n_time)

national_truth_tv = df['effect_tv'].sum() / df['spend_tv'].sum()
national_truth_ps = df['effect_paid_search'].sum() / df['spend_paid_search'].sum()

# Demand proxy quality descriptions (correlation with true baseline bookings)
DEMAND_PROXIES = {
    'demand_perfect':   {'col': 'demand_perfect',   'r': 1.000, 'label': 'Perfect (r=1.00)'},
    'demand_very_good': {'col': 'demand_very_good', 'r': 0.971, 'label': 'Very Good (r=0.97)'},
    'demand_good':      {'col': 'demand_good',      'r': 0.729, 'label': 'Moderate (r=0.73)'},
    'demand_poor':      {'col': 'demand_poor',      'r': 0.110, 'label': 'Poor (r=0.11)'},
    'no_control':       {'col': None,               'r': None,  'label': 'None'},
}



PRIORS = {
    'break_even': {'loc': [0.0, 0.0],                                      'scale': [0.3, 0.3], 'label': 'Break-even (median=1.0)'},
    'tight':      {'loc': [np.log(national_truth_tv), np.log(national_truth_ps)], 'scale': [0.1, 0.1], 'label': 'Tight (median=truth)'},
}

print(f"Ground truth — TV: {national_truth_tv:.3f}, PS: {national_truth_ps:.3f}")
print(f"\nDemand proxy correlations with baseline:")
for k, v in DEMAND_PROXIES.items():
    if v['r']:
        print(f"  {v['label']}")


In [ ]:
def run_meridian(df, prior_config, control_col, key, n_chains=4, n_burnin=500, n_keep=500):
    """Run one Meridian configuration. Returns dict of results."""
    
    meridian_df = df[['date', 'geo', 'population', 'total_bookings',
                       'spend_tv', 'spend_paid_search'] + 
                      ([control_col] if control_col else [])].copy()
    
    csv_path = f'data/tmp_meridian.csv'
    meridian_df.to_csv(csv_path, index=False)
    
    controls = [control_col] if control_col else []
    
    coord_to_columns = load.CoordToColumns(
        time='date', geo='geo', population='population',
        kpi='total_bookings',
        media=['spend_tv', 'spend_paid_search'],
        media_spend=['spend_tv', 'spend_paid_search'],
        controls=controls,
    )
    
    loader = load.CsvDataLoader(
        csv_path=csv_path, kpi_type='revenue',
        coord_to_columns=coord_to_columns,
        media_to_channel={'spend_tv': 'TV', 'spend_paid_search': 'PS'},
        media_spend_to_channel={'spend_tv': 'TV', 'spend_paid_search': 'PS'},
    )
    data = loader.load()
    
    prior = prior_distribution.PriorDistribution(
        roi_m=tfp.distributions.LogNormal(
            loc=tf.constant(prior_config['loc'], dtype=tf.float32),
            scale=tf.constant(prior_config['scale'], dtype=tf.float32),
            name=constants.ROI_M,
        )
    )
    mmm = model.Meridian(input_data=data, model_spec=spec.ModelSpec(prior=prior))
    mmm.sample_posterior(n_chains=n_chains, n_adapt=500,
                     n_burnin=n_burnin, n_keep=n_keep)

    # R² from posterior (no separate predictive sampling needed)

    an = mmm_analyzer.Analyzer(mmm)
    acc = an.predictive_accuracy()
    r2   = float(acc['value'].sel(metric='R_Squared', geo_granularity='national').values)
    mape = float(acc['value'].sel(metric='MAPE',      geo_granularity='national').values)

    # Save InferenceData
    os.makedirs('results', exist_ok=True)
    mmm.inference_data.to_netcdf(f'results/{key}.nc')
    
    posterior = mmm.inference_data.posterior.roi_m
    return {
        'tv_mean':   float(posterior.mean(dim=('chain', 'draw')).values[0]),
        'ps_mean':   float(posterior.mean(dim=('chain', 'draw')).values[1]),
        'tv_median': float(posterior.median(dim=('chain', 'draw')).values[0]),
        'ps_median': float(posterior.median(dim=('chain', 'draw')).values[1]),
        'r2':        r2,
        'mape':      mape,
    }


In [ ]:
results = {}

for prior_name, prior_config in PRIORS.items():
    for proxy_name, proxy_config in DEMAND_PROXIES.items():
        key = f"{prior_name}__{proxy_name}"
        print(f"Running {prior_config['label']} + {proxy_config['label']}...", end=' ', flush=True)
        results[key] = run_meridian(df, prior_config, control_col=proxy_config['col'], key=key)
        print("done")

print(f"\nAll {len(results)} runs complete.")


In [ ]:
print(f"\n{'='*82}")
print(f"ROAS RECOVERY + MODEL FIT  (True TV={national_truth_tv:.2f}, PS={national_truth_ps:.2f})")
print(f"{'='*82}")
print(f"{'Prior':<22} {'Control':<22} {'TV':>7} {'PS':>7} {'TV err':>8} {'PS err':>8} {'MAPE':>7}")
print(f"{'-'*82}")

for prior_name, prior_config in PRIORS.items():
    for proxy_name, proxy_config in DEMAND_PROXIES.items():
        key = f"{prior_name}__{proxy_name}"
        if key not in results:
            continue
        r = results[key]
        tv_err = (r['tv_mean'] - national_truth_tv) / national_truth_tv
        ps_err = (r['ps_mean'] - national_truth_ps) / national_truth_ps
        print(f"{prior_config['label']:<22} {proxy_config['label']:<22} "
              f"{r['tv_mean']:>7.3f} {r['ps_mean']:>7.3f} "
              f"{tv_err:>+7.1%} {ps_err:>+7.1%} "
              f"{r.get('mape', float('nan')):>6.1%}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Build plotting data — exclude no_control (no r value), order by r ascending
proxy_order = ['demand_poor', 'demand_good', 'demand_very_good', 'demand_perfect']
x_labels = [DEMAND_PROXIES[p]['label'] for p in proxy_order]
x = np.arange(len(proxy_order))

prior_styles = {
    'break_even': {'color': '#e15759', 'label': 'Break-even prior (median=1.0)', 'marker': 'o'},
    'tight':      {'color': '#4e79a7', 'label': 'Tight prior (median=truth)',    'marker': 's'},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for prior_name, style in prior_styles.items():
    mapes, avg_errs = [], []
    for proxy_name in proxy_order:
        key = f"{prior_name}__{proxy_name}"
        r = results[key]
        mapes.append(r['mape'] * 100)
        tv_err = abs((r['tv_mean'] - national_truth_tv) / national_truth_tv) * 100
        ps_err = abs((r['ps_mean'] - national_truth_ps) / national_truth_ps) * 100
        avg_errs.append((tv_err + ps_err) / 2)
    
    ax1.plot(x, mapes,    color=style['color'], marker=style['marker'], linewidth=2, markersize=8, label=style['label'])
    ax2.plot(x, avg_errs, color=style['color'], marker=style['marker'], linewidth=2, markersize=8, label=style['label'])

for ax in (ax1, ax2):
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=15, ha='right')
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

ax1.set_title('Model Fit (MAPE)\nlower = better fit', fontsize=12)
ax1.set_ylabel('MAPE (%)')
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

ax2.set_title('ROAS Attribution Error (avg across TV + PS)\nlower = better estimate', fontsize=12)
ax2.set_ylabel('Mean |Error| vs ground truth (%)')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

fig.suptitle('The Fit Paradox: Better Controls → Better Fit → Worse Attribution\n(with miscalibrated priors)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('figures/fit_paradox.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Ground truth and prior medians
import numpy as np
ground_truth = {'TV': national_truth_tv, 'PS': national_truth_ps}
prior_medians = {'break_even': 1.0, 'tight': national_truth_tv}  # tight sets median=truth

channels = ['TV', 'PS']
result_keys = ['tv_mean', 'ps_mean']

for ax, channel, rkey in zip(axes[:2], channels, result_keys):
    for prior_name, style in prior_styles.items():
        estimates = []
        for proxy_name in proxy_order:
            key = f"{prior_name}__{proxy_name}"
            estimates.append(results[key][rkey])
        ax.plot(x, estimates, color=style['color'], marker=style['marker'],
                linewidth=2, markersize=8, label=style['label'])
    
    # Ground truth line
    ax.axhline(ground_truth[channel], color='black', linestyle='--', linewidth=1.5, label=f'Ground truth ({ground_truth[channel]:.2f})')
    # Break-even prior median
    ax.axhline(1.0, color='#e15759', linestyle=':', linewidth=1, alpha=0.6, label='Break-even prior median (1.0)')
    
    ax.set_title(f'{channel} ROAS Estimate', fontsize=12)
    ax.set_ylabel('Estimated ROAS')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=15, ha='right')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# MAPE panel
ax3 = axes[2]
for prior_name, style in prior_styles.items():
    mapes = [results[f"{prior_name}__{p}"]['mape'] * 100 for p in proxy_order]
    ax3.plot(x, mapes, color=style['color'], marker=style['marker'],
             linewidth=2, markersize=8, label=style['label'])
ax3.set_title('Model Fit (MAPE)\nlower = fits data better', fontsize=12)
ax3.set_ylabel('MAPE (%)')
ax3.set_xticks(x)
ax3.set_xticklabels(x_labels, rotation=15, ha='right')
ax3.legend(fontsize=8)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

fig.suptitle('ROAS Estimates vs Ground Truth by Control Quality\n(dashed = ground truth, dotted = break-even prior)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/roas_estimates_vs_truth.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Configuration ──────────────────────────────────────────────────
prior_type   = 'break_even'   # 'tight' | 'break_even'
control      = 'no_control'   # 'no_control' | 'demand_good' | 'demand_very_good' | 'demand_perfect' | 'demand_poor'
# ───────────────────────────────────────────────────────────────────

prior_configs = {
    'tight':      {'loc': [np.log(national_truth_tv), np.log(national_truth_ps)], 'scale': [0.1, 0.1],  'label': 'Tight (median=truth)'},
    'break_even': {'loc': [0.0, 0.0],                                             'scale': [0.3, 0.3],  'label': 'Break-even (median=1.0)'},
}
control_labels = {
    'no_control':      'No Control',
    'demand_good':     'Demand Good (r=0.73)',
    'demand_very_good':'Demand Very Good (r=0.97)',
    'demand_perfect':  'Demand Perfect (r=1.00)',
    'demand_poor':     'Demand Poor (r=0.11)',
}

idata    = az.from_netcdf(f'results/{prior_type}__{control}.nc')
cfg      = prior_configs[prior_type]
title_suffix = f"{cfg['label']} + {control_labels[control]}"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
channels = ['TV', 'Paid_Search']
truths   = [national_truth_tv, national_truth_ps]

for i, (ax, ch, truth) in enumerate(zip(axes, channels, truths)):
    post = idata.posterior['roi_m'].values.reshape(-1, 2)[:, i]
    x    = np.linspace(0, max(post.max(), truth * 2), 300)
    prior_pdf = stats.lognorm.pdf(x, s=cfg['scale'][i], scale=np.exp(cfg['loc'][i]))

    ax.hist(post, bins=50, alpha=0.6, density=True, label='Posterior', color='steelblue')
    ax.plot(x, prior_pdf, 'orange', linewidth=2, label='Prior')
    ax.axvline(truth, color='red', linestyle='--', linewidth=1.5, label=f'Truth ({truth:.2f})')
    ax.set_title(f'{ch} — {title_suffix}')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Configuration ──────────────────────────────────────────────────
prior_type   = 'break_even'   # 'tight' | 'break_even'
control      = 'demand_good'   # 'no_control' | 'demand_good' | 'demand_very_good' | 'demand_perfect' | 'demand_poor'
# ───────────────────────────────────────────────────────────────────

prior_configs = {
    'tight':      {'loc': [np.log(national_truth_tv), np.log(national_truth_ps)], 'scale': [0.1, 0.1],  'label': 'Tight (median=truth)'},
    'break_even': {'loc': [0.0, 0.0],                                             'scale': [0.3, 0.3],  'label': 'Break-even (median=1.0)'},
}
control_labels = {
    'no_control':      'No Control',
    'demand_good':     'Demand Good (r=0.73)',
    'demand_very_good':'Demand Very Good (r=0.97)',
    'demand_perfect':  'Demand Perfect (r=1.00)',
    'demand_poor':     'Demand Poor (r=0.11)',
}

idata    = az.from_netcdf(f'results/{prior_type}__{control}.nc')
cfg      = prior_configs[prior_type]
title_suffix = f"{cfg['label']} + {control_labels[control]}"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
channels = ['TV', 'Paid_Search']
truths   = [national_truth_tv, national_truth_ps]

for i, (ax, ch, truth) in enumerate(zip(axes, channels, truths)):
    post = idata.posterior['roi_m'].values.reshape(-1, 2)[:, i]
    x    = np.linspace(0, max(post.max(), truth * 2), 300)
    prior_pdf = stats.lognorm.pdf(x, s=cfg['scale'][i], scale=np.exp(cfg['loc'][i]))

    ax.hist(post, bins=50, alpha=0.6, density=True, label='Posterior', color='steelblue')
    ax.plot(x, prior_pdf, 'orange', linewidth=2, label='Prior')
    ax.axvline(truth, color='red', linestyle='--', linewidth=1.5, label=f'Truth ({truth:.2f})')
    ax.set_title(f'{ch} — {title_suffix}')
    ax.legend()

plt.tight_layout()
plt.show()
